In [10]:
# Import core libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib
from sklearn.model_selection import train_test_split

# Load dataset
df = pd.read_csv('../data/raw/bank-additional/bank-additional-full.csv', sep=';')
df.head()

,age,job,marital,education,default,housing,loan,contact,month,day_of_week,...,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
0,56,housemaid,married,basic.4y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
1,57,services,married,high.school,unknown,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
2,37,services,married,high.school,no,yes,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
3,40,admin.,married,basic.6y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
4,56,services,married,high.school,no,no,yes,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no


In [11]:
# Separate features and target
X = df.drop('y', axis=1)
y = df['y'].apply(lambda x: 1 if x == 'yes' else 0)

# Stratified split ensures the target class ratio is maintained
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [16]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectFromModel

# Define feature groups
num_cols = ['age', 'campaign', 'pdays', 'previous']
cat_cols = ['job', 'marital', 'education', 'housing', 'loan']

# Create transformer
preprocessor = ColumnTransformer([
    ('num', StandardScaler(), num_cols),
    # request dense output from one-hot encoder to avoid SHAP sparse dtype errors
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols)
])

# Pipeline: Preprocess -> Feature Selection -> Classifier
model_pipeline = Pipeline([
    ('prep', preprocessor),
    ('feature_select', SelectFromModel(RandomForestClassifier(n_estimators=50))),
    ('clf', RandomForestClassifier(n_estimators=100, class_weight='balanced'))
])

In [ ]:
import shap

# Calculate SHAP values
model_pipeline.fit(X_train, y_train)      # ensure pipeline is trained
clf = model_pipeline.named_steps['clf']
explainer = shap.TreeExplainer(clf)      # now clf.estimators_ exists
X_train_transformed = model_pipeline.named_steps['prep'].transform(X_train)
# convert sparse matrix to dense array if necessary
if hasattr(X_train_transformed, "toarray"):
    X_train_transformed = X_train_transformed.toarray()
shap_values = explainer.shap_values(X_train_transformed)

# Plot summary
shap.summary_plot(shap_values, X_train_transformed)

In [ ]:
# Plot for the first lead in the test set
shap.plots.waterfall(shap_values[0])

In [ ]:
# Save the entire pipeline, not just the model
joblib.dump(model_pipeline, 'production_pipeline.pkl')